# Análise Temporal dos Datasets — Dissertação

Este notebook reúne as análises temporais realizadas sobre os datasets utilizados nos experimentos da dissertação, com o objetivo de compreender **por que o HoTHP** (modelo de atenção hiperbólica para processos pontuais temporais) **apresenta desempenho diferente conforme o dataset**.

A hipótese central é que três fatores determinam o sucesso do viés indutivo exponencial do HoTHP:

1. **Alinhamento do kernel** — O kernel de decaimento exponencial do HoTHP precisa estar alinhado com o processo gerador dos dados. Quando o processo real possui auto-excitação com decaimento exponencial (ex.: Hawkes), o HoTHP se beneficia. Quando o kernel real é power-law, o viés é prejudicial.

2. **Calibrabilidade da escala** — Mesmo com kernel parcialmente desalinhado, se os inter-event times (Δt) ocupam uma faixa estreita, o modelo consegue calibrar o parâmetro de escala e compensar.

3. **Suficiência de treinamento** — Datasets maiores permitem que o modelo aprenda a compensar eventuais desalinhamentos do viés indutivo.

As seções a seguir analisam cada fator por meio de estatísticas descritivas, testes de aderência, autocorrelação e ajuste de kernels.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import kstest
from scipy.optimize import curve_fit
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Color scheme by HoTHP outcome
COLOR_WIN  = "#2e7d32"
COLOR_TIE  = "#f57f17"
COLOR_LOSE = "#c62828"

OUTCOME = {
    "hawkes":        "WIN",
    "amazon":        "WIN",
    "taxi":          "TIE",
    "stackoverflow": "LOSE",
    "retweet":       "LOSE",
}

def outcome_color(name):
    o = OUTCOME.get(name, "TIE")
    return {
        "WIN":  COLOR_WIN,
        "TIE":  COLOR_TIE,
        "LOSE": COLOR_LOSE,
    }[o]

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

print("Imports OK", flush=True)

In [ ]:
# ── Dataset loading functions ──

def load_seqs_dts_hf(hf_name, split="train", max_seqs=3000):
    """Load inter-event times (dt) from a HuggingFace dataset.
    Returns a list of numpy arrays, one per sequence."""
    from datasets import load_dataset
    ds = load_dataset(hf_name, split=split)
    seqs = []
    for i, row in enumerate(ds):
        if i >= max_seqs:
            break
        ts = np.array(row["time_since_last_event"], dtype=np.float64)
        # Remove first element (often 0 or padding)
        ts = ts[ts > 0]
        if len(ts) >= 2:
            seqs.append(ts)
    return seqs


def load_times_hf(hf_name, split="train", max_seqs=500):
    """Load absolute timestamps from a HuggingFace dataset.
    Returns a list of numpy arrays, one per sequence."""
    from datasets import load_dataset
    ds = load_dataset(hf_name, split=split)
    seqs = []
    for i, row in enumerate(ds):
        if i >= max_seqs:
            break
        ts = np.array(row["time_since_start"], dtype=np.float64)
        if len(ts) >= 3:
            seqs.append(ts)
    return seqs


def simulate_hawkes_n_events(mu, alpha, beta, n_events, n_seqs, rng):
    """Simulate Hawkes process via Ogata thinning with fixed event count."""
    all_seqs = []
    for _ in range(n_seqs):
        times = [0.0]
        while len(times) < n_events + 1:
            lam_bar = mu + alpha * sum(
                np.exp(-beta * (times[-1] - tj)) for tj in times
            )
            lam_bar = max(lam_bar, 1e-8)
            dt = rng.exponential(1.0 / lam_bar)
            t_cand = times[-1] + dt
            lam_t = mu + alpha * sum(
                np.exp(-beta * (t_cand - tj)) for tj in times
            )
            if rng.uniform() < lam_t / lam_bar:
                times.append(t_cand)
        all_seqs.append(np.array(times[1:]))  # remove t=0
    return all_seqs


def normalize_seqs(event_seqs):
    """Normalize each sequence: compute dt then divide by mean gap.
    Replicates the normalization done in EasyTPP's to_tensors().
    Input: list of absolute-time arrays. Output: list of dt arrays."""
    normed = []
    for ts in event_seqs:
        dts = np.diff(ts)
        dts = dts[dts > 0]
        if len(dts) < 2:
            continue
        mu = dts.mean()
        if mu > 0:
            normed.append(dts / mu)
    return normed


print("Loading functions defined.", flush=True)

In [ ]:
# ── Load all datasets ──

HF_DATASETS = {
    "amazon":        "easytpp/amazon",
    "taxi":          "easytpp/taxi",
    "stackoverflow": "easytpp/stackoverflow",
    "retweet":       "easytpp/retweet",
}

all_seqs_dt = {}   # name -> list of dt arrays
all_times   = {}   # name -> list of absolute time arrays

# ── HuggingFace datasets ──
for name, hf_name in HF_DATASETS.items():
    t0 = time.time()
    print(f"Loading {name}...", end=" ", flush=True)
    all_seqs_dt[name] = load_seqs_dts_hf(hf_name, max_seqs=3000)
    all_times[name]   = load_times_hf(hf_name, max_seqs=500)
    print(f"{len(all_seqs_dt[name])} seqs, {time.time()-t0:.1f}s", flush=True)

# ── Synthetic Hawkes (beta=0.50) ──
print("Simulating Hawkes (beta=0.50)...", end=" ", flush=True)
t0 = time.time()
rng = np.random.default_rng(42)
hawkes_abs = simulate_hawkes_n_events(
    mu=0.2, alpha=0.8, beta=0.50,
    n_events=50, n_seqs=500, rng=rng
)
all_seqs_dt["hawkes"] = normalize_seqs(hawkes_abs)
all_times["hawkes"]   = hawkes_abs
print(f"{len(all_seqs_dt['hawkes'])} seqs, {time.time()-t0:.1f}s", flush=True)

# ── Dataset order for plots ──
DS_ORDER = ["hawkes", "amazon", "taxi", "stackoverflow", "retweet"]

print("\nResumo:", flush=True)
for name in DS_ORDER:
    dts = np.concatenate(all_seqs_dt[name])
    print(f"  {name:15s}: {len(all_seqs_dt[name]):5d} seqs, "
          f"median dt={np.median(dts):.4f}, mean dt={np.mean(dts):.4f}, "
          f"max dt={np.max(dts):.2f}, HoTHP={OUTCOME[name]}", flush=True)

## Seção 1: Análise da Função de Sobrevivência (Survival Function)

O teste de Kolmogorov-Smirnov (KS) mede a distância máxima entre a CDF empírica dos inter-event times normalizados e a CDF de uma distribuição exponencial com taxa unitária.

- **KS ≈ 0** indica que os Δt seguem aproximadamente uma distribuição exponencial — ou seja, o processo é quase Poisson (sem memória temporal).
- **KS >> 0** indica desvio significativo da exponencial, sugerindo auto-excitação, clustering ou outras estruturas temporais.

Para o HoTHP, um KS moderado (indicando estrutura exponencial com auto-excitação) é o cenário ideal, pois o kernel de atenção hiperbólica modela exatamente esse tipo de decaimento.

In [ ]:
# ── Seção 1: Survival Function + KS test ──

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

ks_results = {}

for idx, name in enumerate(DS_ORDER):
    ax = axes[idx]
    # Pool and normalize
    dts = np.concatenate(all_seqs_dt[name])
    dts = dts[dts > 0]
    dts_norm = dts / dts.mean()

    # Empirical survival
    sorted_dt = np.sort(dts_norm)
    survival = 1.0 - np.arange(1, len(sorted_dt) + 1) / len(sorted_dt)

    # KS test vs Exp(1)
    ks_stat, ks_pval = kstest(dts_norm, "expon", args=(0, 1))
    ks_results[name] = ks_stat

    # Reference Exp(1) survival
    x_ref = np.linspace(0, sorted_dt[-1], 500)
    surv_ref = np.exp(-x_ref)

    color = outcome_color(name)
    ax.semilogy(sorted_dt, survival, color=color, alpha=0.7, linewidth=1.5,
                label="Empirical")
    ax.semilogy(x_ref, surv_ref, "k--", alpha=0.5, linewidth=1, label="Exp(1)")
    ax.set_title(f"{name}  KS={ks_stat:.3f}  [{OUTCOME[name]}]", color=color,
                 fontweight="bold")
    ax.set_xlabel("Normalized dt")
    ax.set_ylabel("Survival S(t)")
    ax.set_xlim(0, min(sorted_dt[-1], 8))
    ax.set_ylim(1e-4, 1.1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide unused subplot
axes[-1].set_visible(False)

fig.suptitle("Survival Function Analysis", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig("fig_survival.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print(f"\n{'Dataset':15s} | {'KS stat':>8s} | {'HoTHP':>6s}", flush=True)
print("-" * 38, flush=True)
for name in DS_ORDER:
    print(f"{name:15s} | {ks_results[name]:8.4f} | {OUTCOME[name]:>6s}", flush=True)

## Seção 2: Análise de Autocorrelação (ACF)

A função de autocorrelação (ACF) dos inter-event times mede a **memória temporal** do processo:

- **ACF-1 ≈ 0** indica ausência de correlação serial — o processo é essencialmente sem memória (Poisson-like). Para o HoTHP, isso significa que o viés de decaimento exponencial não encontra estrutura temporal para explorar.
- **ACF-1 > 0** indica clustering temporal — eventos próximos tendem a ser seguidos por mais eventos próximos. Esse padrão é consistente com auto-excitação e favorece o HoTHP.

O **coeficiente de variação (CV)** complementa a análise: CV=1 para Poisson, CV>1 indica sobre-dispersão (clustering), CV<1 indica sub-dispersão (regularidade).

In [ ]:
# ── Seção 2: ACF analysis ──

def acf_multi_lag(seqs_dts, max_lag=8):
    """Weighted-average ACF across sequences (weighted by sequence length)."""
    acf_vals = np.zeros(max_lag)
    total_w  = np.zeros(max_lag)
    for dts in seqs_dts:
        n = len(dts)
        if n < max_lag + 2:
            continue
        mu = dts.mean()
        var = dts.var()
        if var < 1e-15:
            continue
        for lag in range(1, max_lag + 1):
            c = np.mean((dts[:-lag] - mu) * (dts[lag:] - mu))
            acf_vals[lag - 1] += c / var * (n - lag)
            total_w[lag - 1]  += (n - lag)
    mask = total_w > 0
    acf_vals[mask] /= total_w[mask]
    return acf_vals

# Compute ACF and CV for all datasets
acf_results = {}
cv_results  = {}

for name in DS_ORDER:
    acf_results[name] = acf_multi_lag(all_seqs_dt[name], max_lag=8)
    dts_all = np.concatenate(all_seqs_dt[name])
    cv_results[name] = dts_all.std() / dts_all.mean() if dts_all.mean() > 0 else 0.0

# ── Plot 1: ACF bar charts ──
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for idx, name in enumerate(DS_ORDER):
    ax = axes[idx]
    lags = np.arange(1, 9)
    color = outcome_color(name)
    ax.bar(lags, acf_results[name], color=color, alpha=0.8, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"{name}  ACF-1={acf_results[name][0]:.3f}  [{OUTCOME[name]}]",
                 color=color, fontweight="bold")
    ax.set_xlabel("Lag")
    ax.set_ylabel("ACF")
    ax.set_xticks(lags)
    ax.set_ylim(-0.15, max(0.3, acf_results[name].max() * 1.3))
    ax.grid(True, alpha=0.3, axis="y")

axes[-1].set_visible(False)
fig.suptitle("Autocorrelation of Inter-Event Times", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig("fig_acf.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 2: ACF-1 vs CV scatter ──
fig, ax = plt.subplots(figsize=(7, 5))
for name in DS_ORDER:
    color = outcome_color(name)
    ax.scatter(acf_results[name][0], cv_results[name],
               color=color, s=120, edgecolors="black", linewidth=0.8, zorder=5)
    ax.annotate(name, (acf_results[name][0], cv_results[name]),
                textcoords="offset points", xytext=(8, 5), fontsize=9,
                color=color, fontweight="bold")

ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5, label="CV=1 (Poisson)")
ax.axvline(0.0, color="gray", linestyle=":", alpha=0.5, label="ACF-1=0 (no memory)")
ax.set_xlabel("ACF-1 (Temporal Memory)")
ax.set_ylabel("CV (Dispersion)")
ax.set_title("ACF-1 vs Coefficient of Variation", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("fig_acf_cv_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print(f"\n{'Dataset':15s} | {'ACF-1':>8s} | {'CV':>8s} | {'HoTHP':>6s}", flush=True)
print("-" * 48, flush=True)
for name in DS_ORDER:
    print(f"{name:15s} | {acf_results[name][0]:8.4f} | {cv_results[name]:8.4f} | {OUTCOME[name]:>6s}",
          flush=True)

## Seção 3: Ajuste de Kernel (Exponencial vs Power-Law)

A **função de correlação de pares** (pair correlation function) estima a forma do kernel de excitação do processo. Calculamos o histograma das distâncias temporais entre pares de eventos consecutivos dentro de cada sequência, normalizado pela gap média.

Ajustamos dois modelos de kernel:
- **Exponencial**: $g(\Delta t) = \alpha \cdot \exp(-\beta \cdot \Delta t) + c$
- **Power-law**: $g(\Delta t) = \alpha \cdot (1 + \Delta t)^{-p} + c$

Comparamos os ajustes via **AIC** (Akaike Information Criterion). Um ΔAIC negativo (AIC_exp < AIC_pl) indica que o kernel exponencial é melhor, o que favorece o HoTHP.

In [ ]:
# ── Seção 3: Kernel Fit (Exponential vs Power-Law) ──

def pair_correlation(seqs_times, max_pairs_ahead=30, n_bins=80, max_seqs=500):
    """Compute pair correlation function from absolute timestamps.
    Each sequence is normalized by its mean gap before computing pairwise distances."""
    all_dists = []
    for seq_idx, ts in enumerate(seqs_times):
        if seq_idx >= max_seqs:
            break
        if len(ts) < 3:
            continue
        dts = np.diff(ts)
        mean_gap = dts[dts > 0].mean() if np.any(dts > 0) else 1.0
        if mean_gap <= 0:
            continue
        # Normalize timestamps
        ts_norm = (ts - ts[0]) / mean_gap
        for i in range(len(ts_norm)):
            j_end = min(i + max_pairs_ahead + 1, len(ts_norm))
            diffs = ts_norm[i+1:j_end] - ts_norm[i]
            all_dists.extend(diffs.tolist())

    all_dists = np.array(all_dists)
    all_dists = all_dists[all_dists > 0]
    # Clip to reasonable range
    cutoff = np.percentile(all_dists, 95)
    all_dists = all_dists[all_dists <= cutoff]

    counts, edges = np.histogram(all_dists, bins=n_bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, counts


def exp_kernel(dt, alpha, beta, c):
    return alpha * np.exp(-beta * dt) + c

def powerlaw_kernel(dt, alpha, p, c):
    return alpha * (1.0 + dt) ** (-p) + c

def aic(n, rss, k_params):
    if rss <= 0 or n <= 0:
        return np.inf
    return n * np.log(rss / n) + 2 * k_params

def fit_kernels(centers, density):
    """Fit exponential and power-law kernels, return stats."""
    results = {}
    n = len(centers)

    # Exponential fit
    try:
        popt_e, _ = curve_fit(exp_kernel, centers, density,
                              p0=[density.max(), 1.0, density.min()],
                              maxfev=5000, bounds=([0, 0, 0], [np.inf, 50, np.inf]))
        pred_e = exp_kernel(centers, *popt_e)
        rss_e = np.sum((density - pred_e) ** 2)
        ss_tot = np.sum((density - density.mean()) ** 2)
        r2_e = 1.0 - rss_e / ss_tot if ss_tot > 0 else 0.0
        aic_e = aic(n, rss_e, 3)
        results["exp"] = {"popt": popt_e, "pred": pred_e, "r2": r2_e, "aic": aic_e}
    except Exception:
        results["exp"] = {"popt": None, "pred": np.zeros_like(centers),
                          "r2": 0.0, "aic": np.inf}

    # Power-law fit
    try:
        popt_p, _ = curve_fit(powerlaw_kernel, centers, density,
                              p0=[density.max(), 1.0, density.min()],
                              maxfev=5000, bounds=([0, 0, 0], [np.inf, 20, np.inf]))
        pred_p = powerlaw_kernel(centers, *popt_p)
        rss_p = np.sum((density - pred_p) ** 2)
        ss_tot = np.sum((density - density.mean()) ** 2)
        r2_p = 1.0 - rss_p / ss_tot if ss_tot > 0 else 0.0
        aic_p = aic(n, rss_p, 3)
        results["pl"] = {"popt": popt_p, "pred": pred_p, "r2": r2_p, "aic": aic_p}
    except Exception:
        results["pl"] = {"popt": None, "pred": np.zeros_like(centers),
                         "r2": 0.0, "aic": np.inf}

    return results

# ── Compute and plot ──
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
kernel_results = {}

for idx, name in enumerate(DS_ORDER):
    ax = axes[idx]
    centers, density = pair_correlation(all_times[name])
    fits = fit_kernels(centers, density)
    kernel_results[name] = fits

    color = outcome_color(name)
    ax.plot(centers, density, "o", markersize=2, color="gray", alpha=0.5, label="Data")

    if fits["exp"]["popt"] is not None:
        ax.plot(centers, fits["exp"]["pred"], "-", color="blue", linewidth=2,
                alpha=0.8, label=f'Exp R²={fits["exp"]["r2"]:.3f}')
    if fits["pl"]["popt"] is not None:
        ax.plot(centers, fits["pl"]["pred"], "-", color="red", linewidth=2,
                alpha=0.8, label=f'PL R²={fits["pl"]["r2"]:.3f}')

    d_aic = fits["exp"]["aic"] - fits["pl"]["aic"]
    better = "Exp" if d_aic < 0 else "PL"
    ax.set_title(f"{name}\nΔAIC={d_aic:.0f} ({better})\n[{OUTCOME[name]}]",
                 color=color, fontweight="bold", fontsize=9)
    ax.set_xlabel("Normalized Δt")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Pair Correlation Function — Kernel Fit", fontsize=14, fontweight="bold", y=1.05)
fig.tight_layout()
fig.savefig("fig_kernel_fit.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary
print(f"\n{'Dataset':15s} | {'R²_exp':>8s} | {'R²_pl':>8s} | {'ΔAIC':>8s} | {'Better':>7s} | {'HoTHP':>6s}",
      flush=True)
print("-" * 65, flush=True)
for name in DS_ORDER:
    f = kernel_results[name]
    d_aic = f["exp"]["aic"] - f["pl"]["aic"]
    better = "Exp" if d_aic < 0 else "PL"
    print(f"{name:15s} | {f['exp']['r2']:8.4f} | {f['pl']['r2']:8.4f} | {d_aic:8.1f} | {better:>7s} | {OUTCOME[name]:>6s}",
          flush=True)

## Seção 4: Tabela Consolidada

Reunimos as três dimensões de análise em uma tabela única. O framework de três fatores propõe que o HoTHP vence quando:

1. **Kernel alinhado** (ΔAIC < 0, indicando kernel exponencial) 
2. **Escala calibrável** (faixa de Δt estreita, permitindo calibração do parâmetro de escala)
3. **Treinamento suficiente** (volume de dados adequado para aprender os parâmetros)

A combinação desses fatores explica o padrão observado nos resultados experimentais.

In [ ]:
# ── Seção 4: Consolidated Summary ──

# Determine scale calibrability from dt range
scale_info = {}
for name in DS_ORDER:
    dts = np.concatenate(all_seqs_dt[name])
    dts = dts[dts > 0]
    max_dt = np.percentile(dts, 99)
    scale_info[name] = max_dt

# Training size (approximate)
train_sizes = {
    "hawkes": 500,
    "amazon": len(all_seqs_dt.get("amazon", [])),
    "taxi": len(all_seqs_dt.get("taxi", [])),
    "stackoverflow": len(all_seqs_dt.get("stackoverflow", [])),
    "retweet": len(all_seqs_dt.get("retweet", [])),
}

# Print consolidated table
print("=" * 95, flush=True)
print(f"{'Dataset':15s} | {'KS':>6s} | {'ACF-1':>7s} | {'CV':>6s} | {'ΔAIC':>7s} | "
      f"{'Kernel':>7s} | {'Scale':>7s} | {'Train':>6s} | {'HoTHP':>6s}", flush=True)
print("=" * 95, flush=True)

for name in DS_ORDER:
    f = kernel_results[name]
    d_aic = f["exp"]["aic"] - f["pl"]["aic"]
    kernel_winner = "Exp" if d_aic < 0 else "PL"

    # Interpretation markers
    if d_aic < -20:
        kernel_mark = "Exp"
    elif d_aic > 20:
        kernel_mark = "PL"
    else:
        kernel_mark = "~"

    sc = scale_info[name]
    if sc < 2.0:
        scale_mark = "narrow"
    elif sc < 10.0:
        scale_mark = "medium"
    else:
        scale_mark = "wide"

    ts = train_sizes[name]
    if ts >= 2000:
        train_mark = "large"
    elif ts >= 500:
        train_mark = "medium"
    else:
        train_mark = "small"

    print(f"{name:15s} | {ks_results[name]:6.3f} | {acf_results[name][0]:7.4f} | "
          f"{cv_results[name]:6.3f} | {d_aic:7.1f} | {kernel_mark:>7s} | "
          f"{scale_mark:>7s} | {train_mark:>6s} | {OUTCOME[name]:>6s}", flush=True)

print("=" * 95, flush=True)

# ── Summary figure (2x2) ──
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Top-left: KS by dataset
ax = axes[0, 0]
colors = [outcome_color(n) for n in DS_ORDER]
bars = ax.bar(DS_ORDER, [ks_results[n] for n in DS_ORDER], color=colors, edgecolor="white")
ax.set_ylabel("KS Statistic")
ax.set_title("KS Distance from Exponential", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
for bar, name in zip(bars, DS_ORDER):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{ks_results[name]:.3f}", ha="center", va="bottom", fontsize=8)

# Top-right: ACF-1 by dataset
ax = axes[0, 1]
acf1_vals = [acf_results[n][0] for n in DS_ORDER]
bars = ax.bar(DS_ORDER, acf1_vals, color=colors, edgecolor="white")
ax.set_ylabel("ACF-1")
ax.set_title("Autocorrelation at Lag 1", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="y")
for bar, name in zip(bars, DS_ORDER):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{acf_results[name][0]:.3f}", ha="center", va="bottom", fontsize=8)

# Bottom-left: ΔAIC by dataset
ax = axes[1, 0]
daic_vals = [kernel_results[n]["exp"]["aic"] - kernel_results[n]["pl"]["aic"] for n in DS_ORDER]
bars = ax.bar(DS_ORDER, daic_vals, color=colors, edgecolor="white")
ax.set_ylabel("ΔAIC (Exp - PL)")
ax.set_title("Kernel Fit: ΔAIC (negative = Exp better)", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, daic_vals):
    ypos = val + (2 if val >= 0 else -5)
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f"{val:.0f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=8)

# Bottom-right: Three-factor matrix as table
ax = axes[1, 1]
ax.axis("off")

# Build table data
table_data = []
for name in DS_ORDER:
    f = kernel_results[name]
    d_aic = f["exp"]["aic"] - f["pl"]["aic"]

    # Kernel alignment
    if d_aic < -20:
        k_sym = "Exp"
    elif d_aic > 20:
        k_sym = "PL"
    else:
        k_sym = "~"

    sc = scale_info[name]
    if sc < 2.0:
        s_sym = "narrow"
    elif sc < 10.0:
        s_sym = "medium"
    else:
        s_sym = "wide"

    ts = train_sizes[name]
    if ts >= 2000:
        t_sym = "large"
    elif ts >= 500:
        t_sym = "medium"
    else:
        t_sym = "small"

    table_data.append([name, k_sym, s_sym, t_sym, OUTCOME[name]])

col_labels = ["Dataset", "Kernel", "Scale", "Train", "HoTHP"]
table = ax.table(cellText=table_data, colLabels=col_labels,
                 loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)

# Color the HoTHP column
for row_idx in range(len(DS_ORDER)):
    cell = table[row_idx + 1, 4]
    outcome = OUTCOME[DS_ORDER[row_idx]]
    cell.set_facecolor(
        {"WIN": "#c8e6c9", "TIE": "#fff9c4", "LOSE": "#ffcdd2"}[outcome]
    )

ax.set_title("Three-Factor Summary", fontweight="bold", pad=20)

fig.suptitle("Consolidated Analysis Summary", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig("fig_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Conclusões

A análise temporal dos datasets permite explicar o padrão de desempenho do HoTHP observado nos experimentos:

1. **O HoTHP vence quando o kernel exponencial está alinhado com o processo gerador.** No Hawkes sintético, onde o processo gerador possui por construção um kernel de decaimento exponencial, o viés indutivo do HoTHP é perfeitamente alinhado e o modelo supera consistentemente o THP padrão.

2. **A escala limitada dos Δt pode compensar um kernel parcialmente desalinhado.** No Amazon, mesmo que o kernel não seja puramente exponencial, a faixa estreita dos inter-event times normalizados (max Δt ≈ 0.8) permite que o parâmetro de escala do HoTHP se calibre adequadamente, resultando em desempenho superior.

3. **Em processos sem memória temporal (Stackoverflow ≈ Poisson) ou com kernel power-law (Retweet), o viés indutivo monotônico do HoTHP é prejudicial.** O Stackoverflow apresenta ACF-1 próximo de zero e KS elevado sem estrutura de auto-excitação — o decaimento exponencial do HoTHP impõe uma estrutura temporal inexistente. O Retweet possui kernel melhor descrito por power-law, e o decaimento exponencial do HoTHP decai rápido demais para capturar as dependências de longo alcance características desse dataset.

Esses resultados sugerem que a eficácia de modelos com viés indutivo temporal (como o HoTHP) depende criticamente da compatibilidade entre o viés e as propriedades estatísticas do processo gerador dos dados.